In [47]:
import pandas as pd
import sqlite3

fund_master = pd.read_csv("01_fund_master_cleaned.csv")
nav_history = pd.read_csv("02_nav_history_cleaned.csv")
aum = pd.read_csv("03_aum_by_fund_house_cleaned.csv")
sip = pd.read_csv("04_monthly_sip_inflows_cleaned.csv")
category = pd.read_csv("05_category_inflows_cleaned.csv")
folio = pd.read_csv("06_industry_folio_count_cleaned.csv")
performance = pd.read_csv("07_scheme_performance_cleaned.csv")
transactions = pd.read_csv("08_investor_transactions_cleaned.csv")
holdings = pd.read_csv("09_portfolio_holdings_cleaned.csv")
benchmark = pd.read_csv("10_benchmark_indices_cleaned.csv")

In [48]:
import sqlite3

conn = sqlite3.connect("bluestock_mf.db")

In [49]:
fund_master.to_sql(
    "dim_fund",
    conn,
    if_exists="replace",
    index=False
)

40

In [50]:
nav_history.to_sql("fact_nav",conn,if_exists="replace",index=False)

transactions.to_sql("fact_transactions",conn,if_exists="replace",index=False)

performance.to_sql("fact_performance",conn,if_exists="replace",index=False)

aum.to_sql("fact_aum",conn,if_exists="replace",index=False)

benchmark.to_sql("dim_benchmark",conn,if_exists="replace",index=False)

sip.to_sql("fact_sip",conn,if_exists="replace",index=False)

category.to_sql("fact_category",conn,if_exists="replace",index=False)

folio.to_sql("fact_folio",conn,if_exists="replace",index=False)

holdings.to_sql("fact_holdings",conn,if_exists="replace",index=False)

322

In [51]:
cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
""")

print(cursor.fetchall())

[('dim_fund',), ('fact_nav',), ('fact_transactions',), ('fact_performance',), ('fact_aum',), ('dim_benchmark',), ('fact_sip',), ('fact_category',), ('fact_folio',), ('fact_holdings',)]


In [52]:
pd.read_sql(
    "SELECT COUNT(*) FROM dim_fund",
    conn
)

,COUNT(*)
0,40


In [53]:
query = """
SELECT fund_house,
       SUM(aum_crore) AS total_aum
FROM fact_aum
GROUP BY fund_house
ORDER BY total_aum DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,fund_house,total_aum
0,SBI Mutual Fund,8491000
1,ICICI Prudential MF,6293000
2,HDFC Mutual Fund,5732000
3,Nippon India MF,3909000
4,Kotak Mahindra MF,3502000


In [54]:
query = """
SELECT strftime('%Y-%m', date) AS Month,
       AVG(nav) AS Average_NAV
FROM fact_nav
GROUP BY strftime('%Y-%m', date);
"""

pd.read_sql(query, conn)

,Month,Average_NAV
0,2022-01,207.061394
1,2022-02,207.717759
2,2022-03,209.692614
3,2022-04,211.833457
4,2022-05,212.731451
5,2022-06,213.860867
6,2022-07,213.956111
7,2022-08,215.683994
8,2022-09,218.494307
9,2022-10,219.529633


In [55]:
cursor = conn.cursor()

cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('dim_fund',), ('fact_nav',), ('fact_transactions',), ('fact_performance',), ('fact_aum',), ('dim_benchmark',), ('fact_sip',), ('fact_category',), ('fact_folio',), ('fact_holdings',)]


In [56]:
pd.read_sql("PRAGMA table_info(fact_aum);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,date,TEXT,0,None,0
1,1,fund_house,TEXT,0,None,0
2,2,aum_lakh_crore,REAL,0,None,0
3,3,aum_crore,INTEGER,0,None,0
4,4,num_schemes,INTEGER,0,None,0


In [57]:
pd.read_sql("SELECT COUNT(*) FROM dim_fund;", conn)

,COUNT(*)
0,40


In [58]:
pd.read_sql("SELECT COUNT(*) FROM fact_nav;", conn)

,COUNT(*)
0,46000


In [59]:
pd.read_sql("SELECT COUNT(*) FROM fact_aum;", conn)

,COUNT(*)
0,90


In [60]:
pd.read_sql("SELECT COUNT(*) FROM fact_sip;", conn)

,COUNT(*)
0,48


In [61]:
pd.read_sql("SELECT COUNT(*) FROM fact_category;", conn)

,COUNT(*)
0,144


In [62]:
pd.read_sql("SELECT COUNT(*) FROM dim_benchmark;", conn)

,COUNT(*)
0,8050


In [63]:
query = """
SELECT fund_house,
       SUM(aum_crore) AS Total_AUM
FROM fact_aum
GROUP BY fund_house
ORDER BY Total_AUM DESC
LIMIT 5;
"""

pd.read_sql(query, conn)

,fund_house,Total_AUM
0,SBI Mutual Fund,8491000
1,ICICI Prudential MF,6293000
2,HDFC Mutual Fund,5732000
3,Nippon India MF,3909000
4,Kotak Mahindra MF,3502000


In [64]:
query = """
SELECT date,
       SUM(aum_crore) AS Total_AUM
FROM fact_aum
GROUP BY date
ORDER BY date;
"""

pd.read_sql(query, conn)

,date,Total_AUM
0,2022-03-31,3018000
1,2022-09-30,3090000
2,2023-03-31,3235000
3,2023-09-30,3742000
4,2024-03-31,4375000
5,2024-09-30,4727000
6,2024-12-31,5268000
7,2025-03-31,5447000
8,2025-12-31,6274000


In [65]:
query = """
SELECT category,
       ROUND(AVG(return_3yr_pct),2) AS Avg_3Yr_Return
FROM fact_performance
GROUP BY category
ORDER BY Avg_3Yr_Return DESC;
"""

pd.read_sql(query, conn)

,category,Avg_3Yr_Return
0,Small Cap,21.69
1,Mid Cap,16.59
2,Flexi Cap,15.50
3,Value,14.76
4,Large & Mid Cap,14.56
5,ELSS,13.58
6,Large Cap,12.99
7,Index,12.10
8,Index/ETF,11.77
9,Short Duration,7.37


In [66]:
query = """
SELECT scheme_name,
       fund_house,
       expense_ratio_pct
FROM fact_performance
WHERE expense_ratio_pct < 1
ORDER BY expense_ratio_pct;
"""

pd.read_sql(query, conn)

,scheme_name,fund_house,expense_ratio_pct
0,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,0.55
1,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,0.56
2,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,0.60
3,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,0.66
4,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,0.72
5,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,0.72
6,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,0.74
7,Axis Bluechip Fund - Direct - Growth,Axis Mutual Fund,0.75
8,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,0.77
9,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,0.78


In [67]:
query = """
SELECT sector,
       ROUND(AVG(weight_pct),2) AS Average_Weight
FROM fact_holdings
GROUP BY sector
ORDER BY Average_Weight DESC;
"""

pd.read_sql(query, conn)

,sector,Average_Weight
0,Consumer Goods,14.18
1,Diversified,12.09
2,IT,11.39
3,Utilities,11.06
4,FMCG,10.91
5,Banking,10.87
6,NBFC,10.83
7,Pharma,10.72
8,Automobile,9.81
9,Telecom,9.71


In [68]:
query = """
SELECT index_name,
       ROUND(AVG(close_value),2) AS Average_Close
FROM dim_benchmark
GROUP BY index_name
ORDER BY Average_Close DESC;
"""

pd.read_sql(query, conn)

,index_name,Average_Close
0,BSE_SMALLCAP,39375.03
1,NIFTY500,23316.83
2,NIFTY50,22089.36
3,NIFTY_MIDCAP150,22076.53
4,NIFTY100,17186.44
5,CRISIL_LIQUID,2639.93
6,CRISIL_GILT,1779.53


In [69]:
query = """
SELECT transaction_type,
       COUNT(*) AS Total_Transactions
FROM fact_transactions
GROUP BY transaction_type;
"""

pd.read_sql(query, conn)

,transaction_type,Total_Transactions
0,Lumpsum,8095
1,Redemption,4967
2,Sip,19716


In [70]:
query = """
SELECT state,
       SUM(amount_inr) AS Total_Investment
FROM fact_transactions
GROUP BY state
ORDER BY Total_Investment DESC;
"""

pd.read_sql(query, conn)

,state,Total_Investment
0,Punjab,315780459
1,Tamil Nadu,315177237
2,Madhya Pradesh,308312493
3,Rajasthan,298645822
4,Gujarat,298358940
5,West Bengal,297182514
6,Telangana,290219284
7,Delhi,289633404
8,Uttar Pradesh,285368873
9,Haryana,279634354


In [71]:
conn.close()